# Step 7 — Combine LLM-Extracted and Hand-Coded Biographies

Merges two coding sources into a single authoritative CSV:

| File | Source | Rows with data |
|---|---|---|
| `data/personnel_coding/coded_biographies.csv` | LLM (Gemini, Step 5) | 12 |
| `data/personnel_coding/hand_coded_names_qualified_papers.csv` | Manual coding | 12 |

**Merge logic:** both files share the same 12 coded `person_id`s. Hand-coded values are privileged; LLM values fill in only where hand-coded is `NaN`. The output is one row per unique `person_id` (aggregating any multiple-name groups), with all column names normalised to `snake_case`.

**Output:** `data/personnel_coding/combined_coding.csv`

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data/personnel_coding')
OUT_FILE = DATA_DIR / 'combined_coding.csv'

print('Loading source files...')
cb = pd.read_csv(DATA_DIR / 'coded_biographies.csv')
hc = pd.read_csv(DATA_DIR / 'hand_coded_names_qualified_papers.csv', header=1)

print(f'  coded_biographies:             {cb.shape[0]:>4} rows, {cb.shape[1]} cols')
print(f'  hand_coded_names_qualified:    {hc.shape[0]:>4} rows, {hc.shape[1]} cols')
print()
print('coded_biographies columns:', list(cb.columns))
print()
print('hand_coded columns:', list(hc.columns))

Loading source files...
  coded_biographies:               12 rows, 19 cols
  hand_coded_names_qualified:     570 rows, 26 cols

coded_biographies columns: ['research_id', 'canonical_name', 'person_id', 'birthplace', 'education_level', 'family_connection_railroad', 'founded', 'military_service', 'other_corporate_directorships', 'papers_owned_count', 'political_affiliation', 'public_office_sought', 'railroad_board_member', 'railroad_company_owner', 'railroad_professional_services', 'railroad_stockholder', 'wealthy_family', 'year_of_birth', 'year_of_death']

hand_coded columns: ['person_id', 'cleaned_name', 'Unnamed: 2', 'Political Affiliation:', 'Wealthy Family:', 'Birthplace:', 'Ethnicity:', 'Religion:', 'Year of Birth:', 'Year of Death', 'Educ Level', 'Mil Service', 'Public Office Sought', 'Railroad Donation', 'Railroad Stockholder:', 'Railroad Board member:', 'Railroad company owner:', 'Family Connection:', 'Other corporate directorships:', 'Real Estate', 'Wealth At Death', 'Railroad

## 1. Normalise hand-coded columns to snake_case

In [2]:
# Drop the empty helper column
hc = hc.drop(columns=['Unnamed: 2'], errors='ignore')

HC_RENAME = {
    'cleaned_name':                  'name',
    'Political Affiliation:':        'political_affiliation',
    'Wealthy Family:':               'wealthy_family',
    'Birthplace:':                   'birthplace',
    'Ethnicity:':                    'ethnicity',
    'Religion:':                     'religion',
    'Year of Birth:':                'year_of_birth',
    'Year of Death':                 'year_of_death',
    'Educ Level':                    'education_level',
    'Mil Service':                   'military_service',
    'Public Office Sought':          'public_office_sought',
    'Railroad Donation':             'railroad_donation',
    'Railroad Stockholder:':         'railroad_stockholder',
    'Railroad Board member:':        'railroad_board_member',
    'Railroad company owner:':       'railroad_company_owner',
    'Family Connection:':            'family_connection_railroad',
    'Other corporate directorships:':'other_corporate_directorships',
    'Real Estate':                   'real_estate',
    'Wealth At Death':               'wealth_at_death',
    'Railroad Professional Services':'railroad_professional_services',
    'Founded':                       'founded',
    '# of papers owned':             'papers_owned_count',
    'Internal Labor Struggles':      'internal_labor_struggles',
    'Links':                         'links',
}

hc = hc.rename(columns=HC_RENAME)
print('Renamed hand_coded columns:', list(hc.columns))

Renamed hand_coded columns: ['person_id', 'name', 'political_affiliation', 'wealthy_family', 'birthplace', 'ethnicity', 'religion', 'year_of_birth', 'year_of_death', 'education_level', 'military_service', 'public_office_sought', 'railroad_donation', 'railroad_stockholder', 'railroad_board_member', 'railroad_company_owner', 'family_connection_railroad', 'other_corporate_directorships', 'real_estate', 'wealth_at_death', 'railroad_professional_services', 'founded', 'papers_owned_count', 'internal_labor_struggles', 'links']


## 2. Aggregate hand-coded rows to one row per person_id

Multiple names in `names_qualified_papers.csv` can share a `person_id` (they were coded as a group). We collect all names into a `|`-delimited list and take the first non-null value for every data column.

In [3]:
DATA_COLS = [c for c in hc.columns if c not in ('person_id', 'name')]

def first_non_null(series):
    """Return first non-null value in a group, or NaN."""
    vals = series.dropna()
    return vals.iloc[0] if len(vals) else float('nan')

agg_dict = {'name': lambda s: ' | '.join(s.dropna().unique())}
agg_dict.update({col: first_non_null for col in DATA_COLS})

hc_agg = hc.groupby('person_id', as_index=False).agg(agg_dict)

print(f'hand_coded after aggregation: {hc_agg.shape[0]} rows (unique person_ids)')
# Show rows that actually have coded data
has_data = hc_agg[DATA_COLS].notna().any(axis=1)
print(f'  of which {has_data.sum()} have at least one coded field')
print()
print(hc_agg[has_data][['person_id', 'name', 'political_affiliation', 'year_of_birth', 'year_of_death']].to_string(index=False))

hand_coded after aggregation: 463 rows (unique person_ids)
  of which 10 have at least one coded field

 person_id                                                                                                  name political_affiliation  year_of_birth  year_of_death
         3                                                                                         Whitelaw Reid            Republican         1837.0         1912.0
         4 Charles A. Dana | George Ripley | Bayard Taylor | Henry J. Raymond | Horace Greeley | Margaret Fuller            Republican         1819.0         1897.0
        13                                                                                  James Gordon Bennett                   NaN         1841.0         1918.0
        62                                                                                          Michael Hahn            Republican         1830.0         1886.0
       152                                                             

## 3. Normalise coded_biographies columns

`coded_biographies` already uses snake_case. We just rename `canonical_name` → `name` for consistency.

In [4]:
cb_norm = cb.rename(columns={'canonical_name': 'name'}).copy()

# Drop research_id — it's an internal pipeline key not needed in the combined output
cb_norm = cb_norm.drop(columns=['research_id'], errors='ignore')

print('coded_biographies (normalised) columns:', list(cb_norm.columns))
print(f'Rows: {len(cb_norm)}')
cb_norm[['person_id', 'name', 'political_affiliation', 'year_of_birth', 'year_of_death']].head()

coded_biographies (normalised) columns: ['name', 'person_id', 'birthplace', 'education_level', 'family_connection_railroad', 'founded', 'military_service', 'other_corporate_directorships', 'papers_owned_count', 'political_affiliation', 'public_office_sought', 'railroad_board_member', 'railroad_company_owner', 'railroad_professional_services', 'railroad_stockholder', 'wealthy_family', 'year_of_birth', 'year_of_death']
Rows: 12


,person_id,name,political_affiliation,year_of_birth,year_of_death
0,4,Bayard Taylor,NaN,1825.0,1878.0
1,37,Berry R. Sulgrove,Republican,NaN,NaN
2,44,Buckley B. Paddock,NaN,NaN,NaN
3,48,Alfred Doten,NaN,NaN,1903.0
4,103,A. J. Steinman,Democrat,1836.0,1917.0


## 4. Merge — privilege hand-coded, fill gaps with LLM

Strategy:
- Outer-join on `person_id` so all persons from both sources appear.
- For every shared column, use the hand-coded value; fall back to the LLM value only where hand-coded is `NaN`.
- Columns exclusive to hand-coded (ethnicity, religion, railroad_donation, real_estate, wealth_at_death, internal_labor_struggles, links) are kept as-is.
- Track origin in a `source` column.

In [5]:
# Shared data columns (appear in both sources)
SHARED_COLS = [
    'name', 'political_affiliation', 'wealthy_family', 'birthplace',
    'year_of_birth', 'year_of_death', 'education_level', 'military_service',
    'public_office_sought', 'railroad_stockholder', 'railroad_board_member',
    'railroad_company_owner', 'family_connection_railroad',
    'other_corporate_directorships', 'railroad_professional_services',
    'founded', 'papers_owned_count',
]

# Columns only in hand-coded
HC_ONLY_COLS = ['ethnicity', 'religion', 'railroad_donation', 'real_estate',
                'wealth_at_death', 'internal_labor_struggles', 'links']

# Merge
merged = hc_agg.merge(cb_norm, on='person_id', how='outer', suffixes=('_hc', '_llm'))

# Resolve shared columns: prefer _hc, fall back to _llm
for col in SHARED_COLS:
    hc_col  = f'{col}_hc'
    llm_col = f'{col}_llm'
    if hc_col in merged.columns and llm_col in merged.columns:
        merged[col] = merged[hc_col].where(merged[hc_col].notna(), merged[llm_col])
        merged = merged.drop(columns=[hc_col, llm_col])
    elif hc_col in merged.columns:
        merged = merged.rename(columns={hc_col: col})
    elif llm_col in merged.columns:
        merged = merged.rename(columns={llm_col: col})
    # if neither suffix exists the plain column is already there (no collision)

# Add source column
in_hc  = merged['person_id'].isin(hc_agg.loc[hc_agg[DATA_COLS].notna().any(axis=1), 'person_id'])
in_llm = merged['person_id'].isin(cb_norm['person_id'])
merged['source'] = 'none'
merged.loc[in_llm,          'source'] = 'llm_extracted'
merged.loc[in_hc,           'source'] = 'hand_coded'
merged.loc[in_hc & in_llm,  'source'] = 'both'

print(f'Merged shape: {merged.shape}')
print()
print('Source breakdown:')
print(merged['source'].value_counts().to_string())
print()
print('Columns in output:')
print(list(merged.columns))

Merged shape: (463, 26)

Source breakdown:
source
none             442
llm_extracted     11
hand_coded         9
both               1

Columns in output:
['person_id', 'ethnicity', 'religion', 'railroad_donation', 'real_estate', 'wealth_at_death', 'internal_labor_struggles', 'links', 'name', 'political_affiliation', 'wealthy_family', 'birthplace', 'year_of_birth', 'year_of_death', 'education_level', 'military_service', 'public_office_sought', 'railroad_stockholder', 'railroad_board_member', 'railroad_company_owner', 'family_connection_railroad', 'other_corporate_directorships', 'railroad_professional_services', 'founded', 'papers_owned_count', 'source']


## 5. Final column ordering and output

In [7]:
FINAL_COLS = [
    'person_id', 'name', 'source',
    # biographical
    'year_of_birth', 'year_of_death', 'birthplace', 'ethnicity', 'religion',
    'education_level', 'military_service', 'wealthy_family',
    'real_estate', 'wealth_at_death',
    # political / civic
    'political_affiliation', 'public_office_sought', 'internal_labor_struggles',
    # railroad connections
    'railroad_stockholder', 'railroad_board_member', 'railroad_company_owner',
    'railroad_donation', 'railroad_professional_services', 'family_connection_railroad',
    # newspaper / business
    'founded', 'papers_owned_count', 'other_corporate_directorships',
    # meta
    'links',
]

# Keep only the columns we planned for; warn about any extras
extra = [c for c in merged.columns if c not in FINAL_COLS]
if extra:
    print(f'WARNING — unexpected columns not in FINAL_COLS (will be appended): {extra}')
    FINAL_COLS = FINAL_COLS + extra

out = merged[FINAL_COLS].sort_values('person_id').reset_index(drop=True)

out.to_csv(OUT_FILE, index=False)
print(f'Saved {len(out)} rows → {OUT_FILE}')
print()
# Preview coded rows
print('Rows with any coded data:')
coded_cols = [c for c in FINAL_COLS if c not in ('person_id', 'name', 'source', 'links')]
has_coded = out[coded_cols].notna().any(axis=1)
display(out[has_coded][['person_id', 'name', 'source', 'political_affiliation',
                          'year_of_birth', 'year_of_death', 'railroad_board_member']].to_string(index=False))

Saved 463 rows → ..\data\personnel_coding\combined_coding.csv

Rows with any coded data:


' person_id                                                                                                  name        source political_affiliation  year_of_birth  year_of_death  railroad_board_member\n         3                                                                                         Whitelaw Reid    hand_coded            Republican         1837.0         1912.0                    0.0\n         4 Charles A. Dana | George Ripley | Bayard Taylor | Henry J. Raymond | Horace Greeley | Margaret Fuller          both            Republican         1819.0         1897.0                    0.0\n        13                                                                                  James Gordon Bennett    hand_coded                   NaN         1841.0         1918.0                    0.0\n        37                                                                                     Berry R. Sulgrove llm_extracted            Republican            NaN            NaN         